# Corrected cosine and Tversky ontology feature experiment

Primary analysis: unified concept membership tokens; complete ancestor closure; direct ungrouped attribute pairs; Bernoulli feature retention; reference identity excluded from the simulated expression; expected retrieval metrics under random tie resolution. Sensitivity analysis retains reference identity. A legacy reconstruction uses the original representation, fixed-count minimum-one retention, and tie-inclusive metrics on this same March 2026 release. It does not reproduce the earlier September 2025 release.

This is hidden-concept recovery from ontology features, not natural-language normalization or clinical coding validation. Direct attribute pairs do not guarantee semantic containment; audit results are exported. RF2 content and derived concept features remain subject to applicable SNOMED licensing. No clinical correctness labels are manufactured.

Run all cells with Python 3.6+ and pandas, numpy, openpyxl, matplotlib. Set INPUT_DIR and OUT_DIR below. The RF2 ZIP or extracted text is accepted. For a portable rerun from this bundle, the exported concept features and trial plan can replace the licensed RF2-loading stage; see rerun_from_features.py.


In [ ]:
import csv
import hashlib
import json
import math
import os
import platform
import random
import sqlite3
import sys
import time
from datetime import datetime
from functools import lru_cache

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if sys.version_info < (3, 6):
    print("Note: this notebook was prepared for Python 3.6.8; detected:", sys.version)
print("Python:", sys.version.replace("\n", " "))
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("SQLite:", sqlite3.sqlite_version)


In [ ]:
from pathlib import Path
import zipfile
INPUT_DIR = Path(os.environ.get('COSINE_INPUT_DIR', 'inputs')).resolve()
RF2_NAME = 'sct2_Relationship_Full_US1000124_20260301.txt'
SNOMED_REL_FULL_US = str(Path('.') / RF2_NAME)
RF2_ZIP = INPUT_DIR / (RF2_NAME[:-4] + '.zip')
if not os.path.exists(SNOMED_REL_FULL_US):
    with zipfile.ZipFile(str(RF2_ZIP)) as archive:
        with archive.open(RF2_NAME) as source, open(SNOMED_REL_FULL_US, 'wb') as dest:
            import shutil
            shutil.copyfileobj(source, dest)
HSCS_CONCEPT_CODES_XLSX = str(INPUT_DIR / 'HSCS_concept_codes.xlsx')
PAIR_RESULTS_XLSX = None
RUN_STARTED_UTC = datetime.utcnow().isoformat() + "Z"
RUN_ID = "run_" + datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
OUT_ROOT = "./corrected_results"
OUT_DIR = OUT_ROOT
SQLITE_PATH = "./snomed_relationship_snapshot_py36.sqlite"

TERM_COL = "concept_name"
CODE_COL = "concept_code"
LAYPERSON_COL = "style_transfer_conversational_layperson"
EXPECTED_SEED_ROWS = 100

ISA_TYPEID = 116680003
CHARACTERISTIC_TYPEID = 900000000000011006  # Set to a SNOMED characteristic type ID if required.

INCLUDE_ATTRIBUTES = True
INCLUDE_SELF_FEATURE = True
FEATURE_ANCESTOR_DEPTH = 100

# Local-neighborhood design.
TRUE_DESCENDANT_MAX_DEPTH = 3
CANDIDATE_ANCESTOR_DEPTH = 3
CANDIDATE_DESCENDANT_DEPTH = 3
EXTRA_DESCENDANTS_PER_TRIAL = 20
SIBLINGS_PER_PARENT = 10
MAX_CANDIDATES = 40
INVERSION_ANCESTOR_DEPTH = 50
N_TRIALS_PER_TERM = 10

# Manuscript grid.
HIERARCHY_RETENTION_VALUES = (0.4, 0.6, 0.8)
ATTRIBUTE_RETENTION_VALUES = (0.1, 0.3, 0.5)
ALPHA = 1.0
BETAS = (0.1, 0.3, 0.5)

RANDOM_SEED = 7
SCORE_TIE_TOLERANCE = 1e-12

# Cache provenance. Computing a hash for a very large RF2 file is optional.
COMPUTE_RF2_SHA256 = True
REBUILD_CACHE_ON_MISMATCH = True
RF2_INSERT_BATCH_SIZE = 100000
CACHE_SCHEMA_VERSION = "ontology_feature_eval_py36_sqlite_v1"

os.makedirs(OUT_DIR, exist_ok=True)

assert CODE_COL == "concept_code", "Use the curated concept_code column, not an LLM mapping-output column."
assert all(0 < x <= 1 for x in HIERARCHY_RETENTION_VALUES)
assert all(0 < x <= 1 for x in ATTRIBUTE_RETENTION_VALUES)
assert all(0 <= x <= 1 for x in BETAS)


In [ ]:
def sha256_file(path, chunk_size=16 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def file_signature(path, compute_sha256=False):
    absolute = os.path.abspath(path)
    stat = os.stat(absolute)
    result = {
        "path": absolute,
        "size_bytes": int(stat.st_size),
        "mtime_epoch": float(stat.st_mtime),
    }
    if compute_sha256:
        result["sha256"] = sha256_file(absolute)
    return result


def strict_snomed_id(value):
    if pd.isna(value):
        raise ValueError("missing")
    if isinstance(value, (int, np.integer)):
        parsed = int(value)
    elif isinstance(value, (float, np.floating)):
        if not np.isfinite(value) or not float(value).is_integer():
            raise ValueError("not an integer")
        parsed = int(value)
    else:
        text = str(value).strip()
        if not text.isdigit():
            raise ValueError("not a digits-only identifier")
        parsed = int(text)
    if parsed <= 0:
        raise ValueError("identifier must be positive")
    return parsed


def stable_seed(*parts):
    payload = "|".join(str(part) for part in parts).encode("utf-8")
    return int(hashlib.sha256(payload).hexdigest()[:16], 16) % (2 ** 32)


required_paths = [SNOMED_REL_FULL_US, HSCS_CONCEPT_CODES_XLSX]
missing_paths = [path for path in required_paths if not os.path.exists(path)]
if missing_paths:
    raise FileNotFoundError("Update the configuration paths before running: " + ", ".join(missing_paths))

input_signatures = {
    "rf2_relationship_full": file_signature(SNOMED_REL_FULL_US, COMPUTE_RF2_SHA256),
    "hscs_concept_codes": file_signature(HSCS_CONCEPT_CODES_XLSX, True),
}
if PAIR_RESULTS_XLSX and os.path.exists(PAIR_RESULTS_XLSX):
    input_signatures["pair_results"] = file_signature(PAIR_RESULTS_XLSX, True)

raw_source_df = pd.read_excel(
    HSCS_CONCEPT_CODES_XLSX,
    engine="openpyxl",
    nrows=EXPECTED_SEED_ROWS,
    usecols=[TERM_COL, CODE_COL, LAYPERSON_COL],
)
missing_columns = [
    column
    for column in (TERM_COL, CODE_COL)
    if column not in raw_source_df.columns
]
if missing_columns:
    raise KeyError("Missing required columns: " + ", ".join(missing_columns))

def blank_excel_value(value):
    return pd.isna(value) or str(value).strip() == ""


# Excel workbooks can contain formatted/formula tail rows. Ignore a row only
# when both defining fields are blank; never discard a named concept merely
# because its code is missing.
empty_data_row_mask = raw_source_df.apply(
    lambda row: (
        blank_excel_value(row[TERM_COL])
        and blank_excel_value(row[CODE_COL])
    ),
    axis=1,
)
source_df = raw_source_df.loc[~empty_data_row_mask].copy()
dropped_empty_rows = int(empty_data_row_mask.sum())

if len(source_df) != EXPECTED_SEED_ROWS:
    diagnostic_path = os.path.join(
        OUT_DIR,
        "unexpected_concept_workbook_rows.csv",
    )
    source_df[[TERM_COL, CODE_COL]].to_csv(
        diagnostic_path,
        index=True,
    )
    raise ValueError(
        "Expected {} analytical rows in HSCS_concept_codes, but found {} "
        "after removing {} fully blank tail rows. The configured path may "
        "refer to a different workbook. Expected file: the 100-row original "
        "concept-code spreadsheet. Diagnostic: {}".format(
            EXPECTED_SEED_ROWS,
            len(source_df),
            dropped_empty_rows,
            diagnostic_path,
        )
    )

validated_rows = []
invalid_rows = []
for zero_index, row in source_df.iterrows():
    source_row = int(zero_index) + 1
    try:
        root_id = strict_snomed_id(row[CODE_COL])
        validated_rows.append({
            "source_row": source_row,
            "term_id": "HSCS_{:03d}".format(source_row),
            "concept_name": str(row[TERM_COL]).strip(),
            "root_concept_id": root_id,
            "layperson_expression": (
                str(row[LAYPERSON_COL]).strip()
                if LAYPERSON_COL in source_df.columns and not pd.isna(row[LAYPERSON_COL])
                else None
            ),
        })
    except ValueError as exc:
        invalid_rows.append({
            "source_row": source_row,
            "concept_name": row.get(TERM_COL),
            "raw_code": row.get(CODE_COL),
            "reason": str(exc),
        })

terms_df = pd.DataFrame(validated_rows)
terms_df = terms_df.drop_duplicates(
    subset=["root_concept_id"]
).copy()
invalid_codes_df = pd.DataFrame(invalid_rows)
if invalid_rows:
    invalid_codes_df.to_csv(os.path.join(OUT_DIR, "invalid_seed_codes.csv"), index=False)
    raise ValueError("{} invalid concept_code values; see invalid_seed_codes.csv".format(len(invalid_rows)))

terms_df.to_csv(os.path.join(OUT_DIR, "validated_seed_concepts.csv"), index=False)
print("Input rows:", len(source_df))
print("Ignored fully blank Excel tail rows:", dropped_empty_rows)
print("Valid authoritative concept_code values:", len(terms_df))
print("Unique seed concepts:", terms_df["root_concept_id"].nunique())
print("Duplicate seed rows excluded:", len(source_df) - len(terms_df))
terms_df.head()


In [ ]:
def sqlite_table_names(connection):
    return {
        row[0]
        for row in connection.execute(
            "SELECT name FROM sqlite_master WHERE type = 'table'"
        ).fetchall()
    }


def configure_sqlite_for_build(connection):
    connection.execute("PRAGMA journal_mode = OFF")
    connection.execute("PRAGMA synchronous = OFF")
    connection.execute("PRAGMA temp_store = FILE")
    connection.execute("PRAGMA cache_size = -262144")


def rf2_integer(value, field_name, row_number):
    text = str(value).strip()
    if text == "":
        if field_name == "effectiveTime":
            return 0
        raise ValueError(
            "Blank {} at RF2 row {}".format(field_name, row_number)
        )
    try:
        return int(text)
    except ValueError:
        raise ValueError(
            "Invalid {}={!r} at RF2 row {}".format(
                field_name, value, row_number
            )
        )


expected_cache_metadata = {
    "cache_schema_version": CACHE_SCHEMA_VERSION,
    "rf2_path": input_signatures["rf2_relationship_full"]["path"],
    "rf2_size_bytes": str(input_signatures["rf2_relationship_full"]["size_bytes"]),
    "rf2_mtime_epoch": str(input_signatures["rf2_relationship_full"]["mtime_epoch"]),
    "characteristic_type_id": str(CHARACTERISTIC_TYPEID),
}
if "sha256" in input_signatures["rf2_relationship_full"]:
    expected_cache_metadata["rf2_sha256"] = input_signatures["rf2_relationship_full"]["sha256"]

con = None
tables = set()
cached_metadata = {}
if os.path.exists(SQLITE_PATH):
    con = sqlite3.connect(SQLITE_PATH)
    tables = sqlite_table_names(con)
    if "cache_metadata" in tables:
        cached_metadata = dict(
            con.execute("SELECT key, value FROM cache_metadata").fetchall()
        )

cache_matches = (
    "rel_snap" in tables
    and all(str(cached_metadata.get(key)) == str(value)
            for key, value in expected_cache_metadata.items())
)

if "rel_snap" in tables and not cache_matches and not REBUILD_CACHE_ON_MISMATCH:
    raise RuntimeError(
        "Relationship cache provenance mismatch. "
        "Set REBUILD_CACHE_ON_MISMATCH=True or use a new SQLITE_PATH."
    )

if not cache_matches:
    if con is not None:
        con.close()
        con = None

    print(
        "Building the SQLite relationship snapshot. "
        "This is a one-time operation and may take a while."
    )
    build_path = SQLITE_PATH + ".building"
    if os.path.exists(build_path):
        os.remove(build_path)

    build_con = sqlite3.connect(build_path)
    configure_sqlite_for_build(build_con)
    build_con.execute(
        '''
        CREATE TABLE rel_raw (
            id TEXT NOT NULL,
            effectiveTime INTEGER NOT NULL,
            active INTEGER NOT NULL,
            sourceId INTEGER NOT NULL,
            destinationId INTEGER NOT NULL,
            relationshipGroup INTEGER NOT NULL,
            typeId INTEGER NOT NULL,
            characteristicTypeId INTEGER NOT NULL
        )
        '''
    )

    insert_sql = (
        "INSERT INTO rel_raw "
        "(id, effectiveTime, active, sourceId, destinationId, "
        "relationshipGroup, typeId, characteristicTypeId) "
        "VALUES (?, ?, ?, ?, ?, ?, ?, ?)"
    )
    started = time.time()
    loaded_rows = 0
    batch = []
    with open(
        SNOMED_REL_FULL_US,
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as rf2_handle:
        reader = csv.reader(rf2_handle, delimiter="\t")
        try:
            header = next(reader)
        except StopIteration:
            raise ValueError("RF2 relationship file is empty")

        required_rf2_columns = [
            "id",
            "effectiveTime",
            "active",
            "sourceId",
            "destinationId",
            "relationshipGroup",
            "typeId",
            "characteristicTypeId",
        ]
        header_index = {
            str(column).strip(): position
            for position, column in enumerate(header)
        }
        missing_rf2_columns = [
            column
            for column in required_rf2_columns
            if column not in header_index
        ]
        if missing_rf2_columns:
            raise KeyError(
                "RF2 file is missing columns: "
                + ", ".join(missing_rf2_columns)
            )

        for row_number, row in enumerate(reader, start=2):
            batch.append((
                str(row[header_index["id"]]).strip(),
                rf2_integer(
                    row[header_index["effectiveTime"]],
                    "effectiveTime",
                    row_number,
                ),
                rf2_integer(
                    row[header_index["active"]], "active", row_number
                ),
                rf2_integer(
                    row[header_index["sourceId"]], "sourceId", row_number
                ),
                rf2_integer(
                    row[header_index["destinationId"]],
                    "destinationId",
                    row_number,
                ),
                rf2_integer(
                    row[header_index["relationshipGroup"]],
                    "relationshipGroup",
                    row_number,
                ),
                rf2_integer(
                    row[header_index["typeId"]], "typeId", row_number
                ),
                rf2_integer(
                    row[header_index["characteristicTypeId"]],
                    "characteristicTypeId",
                    row_number,
                ),
            ))
            if len(batch) >= RF2_INSERT_BATCH_SIZE:
                build_con.executemany(insert_sql, batch)
                build_con.commit()
                loaded_rows += len(batch)
                batch = []
                if loaded_rows % 1000000 == 0:
                    print(
                        "Loaded {:,} RF2 rows in {:.1f} minutes".format(
                            loaded_rows, (time.time() - started) / 60.0
                        )
                    )

        if batch:
            build_con.executemany(insert_sql, batch)
            build_con.commit()
            loaded_rows += len(batch)

    print("Indexing {:,} RF2 rows...".format(loaded_rows))
    build_con.execute(
        "CREATE INDEX idx_rel_raw_id_time "
        "ON rel_raw(id, effectiveTime)"
    )
    build_con.commit()

    characteristic_filter = ""
    parameters = []
    if CHARACTERISTIC_TYPEID is not None:
        characteristic_filter = " AND r.characteristicTypeId = ?"
        parameters.append(int(CHARACTERISTIC_TYPEID))

    build_con.execute(
        '''
        CREATE TABLE rel_snap AS
        SELECT
            r.sourceId AS sourceId,
            r.typeId AS typeId,
            r.destinationId AS destinationId,
            r.relationshipGroup AS relationshipGroup,
            r.characteristicTypeId AS characteristicTypeId
        FROM rel_raw AS r
        INNER JOIN (
            SELECT id, MAX(effectiveTime) AS latest_effective_time
            FROM rel_raw
            GROUP BY id
        ) AS latest
          ON r.id = latest.id
         AND r.effectiveTime = latest.latest_effective_time
        WHERE r.active = 1
        ''' + characteristic_filter,
        parameters,
    )
    build_con.execute(
        "CREATE INDEX idx_rel_snap_src ON rel_snap(sourceId)"
    )
    build_con.execute(
        "CREATE INDEX idx_rel_snap_type ON rel_snap(typeId)"
    )
    build_con.execute(
        "CREATE INDEX idx_rel_snap_dst ON rel_snap(destinationId)"
    )
    build_con.execute(
        "CREATE TABLE cache_metadata(key TEXT, value TEXT)"
    )
    build_con.executemany(
        "INSERT INTO cache_metadata VALUES (?, ?)",
        [(str(key), str(value)) for key, value in sorted(expected_cache_metadata.items())],
    )
    build_con.execute("DROP TABLE rel_raw")
    build_con.commit()
    build_con.execute("VACUUM")
    build_con.close()

    os.replace(build_path, SQLITE_PATH)
    con = sqlite3.connect(SQLITE_PATH)
    print(
        "SQLite snapshot built in {:.1f} minutes.".format(
            (time.time() - started) / 60.0
        )
    )
else:
    print("Reusing the provenance-matched SQLite relationship snapshot.")

print("Active relationship rows:", con.execute("SELECT COUNT(*) FROM rel_snap").fetchone()[0])

# Composite indexes avoid repeated scans during graph traversal.
con.execute("CREATE INDEX IF NOT EXISTS idx_rel_src_type_dst ON rel_snap(sourceId,typeId,destinationId)")
con.execute("CREATE INDEX IF NOT EXISTS idx_rel_dst_type_src ON rel_snap(destinationId,typeId,sourceId)")
con.execute("ANALYZE")
con.commit()


In [ ]:
@lru_cache(maxsize=500000)
def parents_of(concept_id):
    rows = con.execute(
        "SELECT DISTINCT destinationId FROM rel_snap WHERE sourceId = ? AND typeId = ?",
        [int(concept_id), int(ISA_TYPEID)],
    ).fetchall()
    return tuple(sorted(int(row[0]) for row in rows))


@lru_cache(maxsize=500000)
def children_of(concept_id):
    rows = con.execute(
        "SELECT DISTINCT sourceId FROM rel_snap WHERE destinationId = ? AND typeId = ?",
        [int(concept_id), int(ISA_TYPEID)],
    ).fetchall()
    return tuple(sorted(int(row[0]) for row in rows))


@lru_cache(maxsize=500000)
def non_isa_relationships(concept_id):
    if not INCLUDE_ATTRIBUTES:
        return tuple()
    rows = con.execute(
        "SELECT DISTINCT typeId, destinationId FROM rel_snap "
        "WHERE sourceId = ? AND typeId <> ?",
        [int(concept_id), int(ISA_TYPEID)],
    ).fetchall()
    return tuple(sorted((int(row[0]), int(row[1])) for row in rows))


@lru_cache(maxsize=500000)
def bounded_ancestors(concept_id, max_depth):
    seen = set()
    frontier = {int(concept_id)}
    for _ in range(int(max_depth)):
        next_frontier = set()
        for node in sorted(frontier):
            next_frontier.update(parents_of(node))
        next_frontier.difference_update(seen)
        if not next_frontier:
            break
        seen.update(next_frontier)
        frontier = next_frontier
    seen.discard(int(concept_id))
    return frozenset(seen)


@lru_cache(maxsize=500000)
def bounded_descendants(concept_id, max_depth):
    seen = set()
    frontier = {int(concept_id)}
    for _ in range(int(max_depth)):
        next_frontier = set()
        for node in sorted(frontier):
            next_frontier.update(children_of(node))
        next_frontier.difference_update(seen)
        if not next_frontier:
            break
        seen.update(next_frontier)
        frontier = next_frontier
    seen.discard(int(concept_id))
    return frozenset(seen)


def concept_in_relationship_snapshot(concept_id):
    count = con.execute(
        "SELECT COUNT(*) FROM rel_snap WHERE sourceId = ? OR destinationId = ?",
        [int(concept_id), int(concept_id)],
    ).fetchone()[0]
    return int(count) > 0


@lru_cache(maxsize=500000)
def hierarchy_features(concept_id):
    features = set()
    if INCLUDE_SELF_FEATURE:
        features.add("self:{}".format(int(concept_id)))
    for ancestor_id in bounded_ancestors(int(concept_id), FEATURE_ANCESTOR_DEPTH):
        features.add("isa:{}".format(int(ancestor_id)))
    return frozenset(features)


@lru_cache(maxsize=500000)
def attribute_features(concept_id):
    return frozenset(
        "rel:{}={}".format(type_id, destination_id)
        for type_id, destination_id in non_isa_relationships(int(concept_id))
    )


@lru_cache(maxsize=500000)
def concept_features(concept_id):
    return frozenset(
        set(hierarchy_features(int(concept_id))) |
        set(attribute_features(int(concept_id)))
    )


In [ ]:
def cosine_kec(k, e, c):
    k, e, c = float(k), float(e), float(c)
    denominator = math.sqrt((k + e) * (k + c))
    return 0.0 if denominator == 0 else k / denominator


def tversky_kec(k, e, c, alpha=1.0, beta=0.3):
    k, e, c = float(k), float(e), float(c)
    denominator = k + float(alpha) * e + float(beta) * c
    return 0.0 if denominator == 0 else k / denominator


def kec(expression_features, concept_feature_set):
    expression_features = set(expression_features)
    concept_feature_set = set(concept_feature_set)
    return (
        len(expression_features & concept_feature_set),
        len(expression_features - concept_feature_set),
        len(concept_feature_set - expression_features),
    )


def competition_ranks(scores, tolerance=SCORE_TIE_TOLERANCE):
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")
    ranks = np.empty(len(scores), dtype=int)
    previous_score = None
    current_rank = None
    for position, original_index in enumerate(order):
        score = float(scores[original_index])
        if previous_score is None or not math.isclose(
            score, previous_score, rel_tol=tolerance, abs_tol=tolerance
        ):
            current_rank = position + 1
            previous_score = score
        ranks[original_index] = current_rank
    return ranks


def retain_features(features, fraction, seed):
    ordered = sorted(set(features))
    if not ordered:
        return frozenset()
    rng = random.Random(int(seed))
    rng.shuffle(ordered)
    keep_count = max(1, min(len(ordered), int(round(float(fraction) * len(ordered)))))
    return frozenset(ordered[:keep_count])


def simulate_expression(true_concept_id, hierarchy_retention, attribute_retention, seed):
    full_hierarchy = hierarchy_features(int(true_concept_id))
    full_attributes = attribute_features(int(true_concept_id))
    kept_hierarchy = retain_features(
        full_hierarchy,
        hierarchy_retention,
        stable_seed(seed, "hierarchy-order"),
    )
    kept_attributes = (
        retain_features(
            full_attributes,
            attribute_retention,
            stable_seed(seed, "attribute-order"),
        )
        if INCLUDE_ATTRIBUTES else frozenset()
    )
    expression = frozenset(set(kept_hierarchy) | set(kept_attributes))
    metadata = {
        "hierarchy_total": len(full_hierarchy),
        "hierarchy_kept": len(kept_hierarchy),
        "attribute_total": len(full_attributes),
        "attribute_kept": len(kept_attributes),
        "expression_feature_count": len(expression),
    }
    return expression, metadata


def capped_candidates(mandatory_ids, optional_ids, maximum, seed):
    mandatory = sorted(set(int(x) for x in mandatory_ids))
    optional = sorted(set(int(x) for x in optional_ids) - set(mandatory))
    if len(mandatory) >= int(maximum):
        # Scientific invariants take priority over an approximate cap.
        return tuple(mandatory)
    rng = random.Random(int(seed))
    remaining = int(maximum) - len(mandatory)
    if len(optional) > remaining:
        optional = rng.sample(optional, remaining)
    return tuple(sorted(mandatory + optional))


# Self-contained tests. These run before any long simulation.
assert math.isclose(cosine_kec(3, 0, 1), 3 / math.sqrt(12), rel_tol=1e-12)
assert math.isclose(tversky_kec(3, 0, 1, 1.0, 0.3), 3 / 3.3, rel_tol=1e-12)
assert competition_ranks([0.9, 0.9, 0.5]).tolist() == [1, 1, 3]
_retained_sparse = retain_features({"a", "b", "c", "d", "e"}, 0.4, 1)
_retained_rich = retain_features({"a", "b", "c", "d", "e"}, 0.8, 1)
assert len(_retained_sparse) == 2
assert _retained_sparse.issubset(_retained_rich)
_test_candidates = capped_candidates({1, 2, 3}, range(1, 50), 10, 7)
assert {1, 2, 3}.issubset(_test_candidates) and len(_test_candidates) == 10
print("Formula, ranking, retention, and candidate-preservation tests passed.")


In [ ]:
def build_candidate_set(root_id, true_id, seed):
    relevant_ancestors = set(
        bounded_ancestors(int(true_id), CANDIDATE_ANCESTOR_DEPTH)
    )
    mandatory = {int(root_id), int(true_id)} | relevant_ancestors

    rng = random.Random(stable_seed(seed, "optional-neighborhood"))
    descendants = sorted(
        bounded_descendants(int(true_id), CANDIDATE_DESCENDANT_DEPTH)
    )
    if len(descendants) > EXTRA_DESCENDANTS_PER_TRIAL:
        descendants = rng.sample(descendants, EXTRA_DESCENDANTS_PER_TRIAL)

    siblings = set()
    for parent_id in parents_of(int(true_id)):
        parent_children = [x for x in children_of(parent_id) if int(x) != int(true_id)]
        if len(parent_children) > SIBLINGS_PER_PARENT:
            parent_children = rng.sample(parent_children, SIBLINGS_PER_PARENT)
        siblings.update(parent_children)

    optional = set(descendants) | siblings
    candidates = capped_candidates(
        mandatory,
        optional,
        MAX_CANDIDATES,
        stable_seed(seed, "candidate-cap"),
    )
    assert int(true_id) in candidates
    assert int(root_id) in candidates
    assert relevant_ancestors.issubset(set(candidates))
    return candidates


trial_plan_rows = []
excluded_seed_rows = []

for _, term in terms_df.iterrows():
    root_id = int(term["root_concept_id"])
    if not concept_in_relationship_snapshot(root_id):
        excluded_seed_rows.append({
            "source_row": int(term["source_row"]),
            "term_id": term["term_id"],
            "concept_name": term["concept_name"],
            "root_concept_id": root_id,
            "reason": "seed_not_found_in_relationship_snapshot",
        })
        continue

    eligible_descendants = sorted(
        bounded_descendants(root_id, TRUE_DESCENDANT_MAX_DEPTH)
    )
    if not eligible_descendants:
        excluded_seed_rows.append({
            "source_row": int(term["source_row"]),
            "term_id": term["term_id"],
            "concept_name": term["concept_name"],
            "root_concept_id": root_id,
            "reason": "no_descendant_within_depth_{}".format(TRUE_DESCENDANT_MAX_DEPTH),
        })
        continue

    for trial_number in range(1, N_TRIALS_PER_TERM + 1):
        trial_seed = stable_seed(
            RANDOM_SEED, int(term["source_row"]), root_id, trial_number
        )
        rng = random.Random(trial_seed)
        true_id = int(rng.choice(eligible_descendants))
        candidate_ids = build_candidate_set(root_id, true_id, trial_seed)
        inversion_ancestors = sorted(
            set(bounded_ancestors(true_id, INVERSION_ANCESTOR_DEPTH))
            & set(candidate_ids)
        )
        trial_plan_rows.append({
            "trial_id": "{}_T{:02d}".format(term["term_id"], trial_number),
            "source_row": int(term["source_row"]),
            "term_id": term["term_id"],
            "concept_name": term["concept_name"],
            "root_concept_id": root_id,
            "true_concept_id": true_id,
            "trial_number": trial_number,
            "trial_seed": trial_seed,
            "descendant_pool_size": len(eligible_descendants),
            "n_candidates": len(candidate_ids),
            "candidate_ids_json": json.dumps(list(candidate_ids)),
            "ancestor_candidate_ids_json": json.dumps(inversion_ancestors),
        })

trial_plan_df = pd.DataFrame(trial_plan_rows)
excluded_seeds_df = pd.DataFrame(
    excluded_seed_rows,
    columns=[
        "source_row",
        "term_id",
        "concept_name",
        "root_concept_id",
        "reason",
    ],
)
if trial_plan_df.empty:
    raise RuntimeError(
        "No eligible trials were created. Check the RF2 edition, "
        "seed identifiers, and TRUE_DESCENDANT_MAX_DEPTH."
    )
trial_plan_df.to_csv(os.path.join(OUT_DIR, "trial_plan.csv"), index=False)
excluded_seeds_df.to_csv(os.path.join(OUT_DIR, "excluded_seed_rows.csv"), index=False)

print("Eligible source rows:", trial_plan_df["source_row"].nunique())
print("Planned trials per condition:", len(trial_plan_df))
print("Excluded source rows:", len(excluded_seeds_df))
print("Mean candidates per trial:", trial_plan_df["n_candidates"].mean())
if len(excluded_seeds_df):
    display(excluded_seeds_df["reason"].value_counts(dropna=False))
else:
    print("No seed rows were excluded.")


## Corrected representations and paired evaluation

All scenarios reuse the same reference concepts and candidates. Full ancestor closure is used in corrected scenarios, with one namespace for self and ancestor identities. Attributes remain direct type-destination pairs and are explicitly treated as a simplification. Bernoulli draws share random uniforms across retention settings, so retained sets are nested. Empty expressions are retained and flagged, with all scores zero and expected metrics reflecting uninformative ties. Primary metrics average over random ordering within tied scores; legacy tie-inclusive scores are also exported. Parent-over-child means an ancestor has a strictly greater score, not a proven clinical error.


In [ ]:
# Build feature export covering every evaluated candidate.
used_ids = sorted(set(x for v in trial_plan_df.candidate_ids_json for x in json.loads(v)))
@lru_cache(maxsize=None)
def full_ancestors(cid):
    seen = set(); frontier = {int(cid)}
    while frontier:
        nxt = set()
        for x in frontier: nxt.update(parents_of(x))
        nxt -= seen
        seen.update(nxt); frontier = nxt
    seen.discard(int(cid))
    return frozenset(seen)

feature_export = {}
for cid in used_ids:
    feature_export[str(cid)] = {
        'hierarchy': sorted({'concept:{}'.format(x) for x in set(full_ancestors(cid)) | {cid}}),
        'attributes': sorted(attribute_features(cid)),
        'legacy_hierarchy': sorted({'self:{}'.format(cid)} | {'isa:{}'.format(x) for x in bounded_ancestors(cid,15)}),
    }
with open(os.path.join(OUT_DIR,'concept_features.json'),'w') as f: json.dump(feature_export,f,sort_keys=True)
# Audit characteristic types across the latest active snapshot actually selected.
print('Characteristic types:', con.execute('SELECT characteristicTypeId, COUNT(*) FROM rel_snap GROUP BY characteristicTypeId').fetchall())
print('Unique evaluated candidate concepts:',len(used_ids))
con.close()


In [ ]:
import gzip
SCENARIOS = ('primary_no_identity','identity_retained','legacy_same_release')
METHODS = ('cosine','tversky_0.1','tversky_0.3','tversky_0.5','jaccard','dice')
BOOTSTRAPS = 2000

def scores_for(counts, method):
    if method == 'cosine': return [cosine_kec(*x) for x in counts]
    if method == 'jaccard': return [tversky_kec(*x,alpha=1,beta=1) for x in counts]
    if method == 'dice': return [tversky_kec(*x,alpha=.5,beta=.5) for x in counts]
    return [tversky_kec(*x,alpha=1,beta=float(method.split('_')[1])) for x in counts]

def retrieval_metrics(scores, true_index):
    arr = np.array(scores); target = arr[true_index]
    equal = np.isclose(arr,target,rtol=SCORE_TIE_TOLERANCE,atol=SCORE_TIE_TOLERANCE)
    greater = int(((arr > target) & ~equal).sum()); tied = int(equal.sum())
    return {'top1': (1.0/tied if greater == 0 else 0.0),
            'top5': max(0.0,min(1.0,(5.0-greater)/tied)),
            'mrr': sum(1.0/r for r in range(greater+1,greater+tied+1))/tied,
            'tie_size':tied,'optimistic_top1':int(greater==0),
            'optimistic_top5':int(greater<5),'optimistic_mrr':1.0/(greater+1)}
assert retrieval_metrics([1]*10,0)['top1'] == .1
assert retrieval_metrics([1]*10,0)['top5'] == .5
assert math.isclose(retrieval_metrics([1,1],0)['mrr'],.75)
assert retain_features(set(range(2)),.1,7)  # legacy behavior explicitly preserved only in legacy scenario

# Theory audit is descriptive: direct attributes need not be inherited.
audit_rows=[]
for _, plan in trial_plan_df.drop_duplicates(['root_concept_id','true_concept_id']).iterrows():
    root=int(plan.root_concept_id); child=int(plan.true_concept_id)
    p=feature_export[str(root)]; q=feature_export[str(child)]
    ph=set(p['hierarchy']); ch=set(q['hierarchy']); pa=set(p['attributes']); ca=set(q['attributes'])
    audit_rows.append({'root_concept_id':root,'reference_concept_id':child,
                      'hierarchy_contained':ph <= ch,'full_features_contained':(ph|pa) <= (ch|ca),
                      'missing_parent_attributes':len(pa-ca)})
audit_df=pd.DataFrame(audit_rows); audit_df.to_csv(os.path.join(OUT_DIR,'feature_containment_audit.csv'),index=False)
assert audit_df.hierarchy_contained.all()

trial_rows=[]; expression_rows=[]
score_path=os.path.join(OUT_DIR,'candidate_scores.csv.gz')
score_handle=gzip.open(score_path,'wt',newline=''); writer=None
for ix,plan in trial_plan_df.iterrows():
    ids=json.loads(plan.candidate_ids_json); ti=ids.index(int(plan.true_concept_id)); ref=feature_export[str(int(plan.true_concept_id))]
    ancestor_ids=set(json.loads(plan.ancestor_candidate_ids_json))
    expr_seed=stable_seed(int(plan.trial_seed),'expression-order')
    for scenario in SCENARIOS:
        legacy=scenario=='legacy_same_release'
        hier_key='legacy_hierarchy' if legacy else 'hierarchy'
        hier=set(ref[hier_key]); attrs=set(ref['attributes'])
        if scenario=='primary_no_identity': hier.discard('concept:{}'.format(int(plan.true_concept_id)))
        # Uniforms are shared across conditions and corrected scenarios.
        hierarchy_uniform={f:random.Random(stable_seed(expr_seed,'bernoulli',f)).random() for f in hier}
        attribute_uniform={f:random.Random(stable_seed(expr_seed,'bernoulli',f)).random() for f in attrs}
        for hr in HIERARCHY_RETENTION_VALUES:
            for ar in ATTRIBUTE_RETENTION_VALUES:
                condition='H{:.1f}_A{:.1f}'.format(hr,ar)
                if legacy:
                    kh=retain_features(hier,hr,stable_seed(expr_seed,'hierarchy-order'))
                    ka=retain_features(attrs,ar,stable_seed(expr_seed,'attribute-order'))
                else:
                    kh={f for f,u in hierarchy_uniform.items() if u<hr}
                    ka={f for f,u in attribute_uniform.items() if u<ar}
                expr=set(kh)|set(ka)
                csets=[set(feature_export[str(cid)][hier_key])|set(feature_export[str(cid)]['attributes']) for cid in ids]
                counts=[kec(expr,cset) for cset in csets]
                rootset=set(feature_export[str(int(plan.root_concept_id))][hier_key])|set(feature_export[str(int(plan.root_concept_id))]['attributes'])
                meta={'trial_id':plan.trial_id,'source_row':int(plan.source_row),'root_concept_id':int(plan.root_concept_id),
                      'true_concept_id':int(plan.true_concept_id),'scenario':scenario,'condition_id':condition,
                      'hierarchy_retention':hr,'attribute_retention':ar,'n_candidates':len(ids),
                      'hierarchy_total':len(hier),'hierarchy_kept':len(kh),'attribute_total':len(attrs),'attribute_kept':len(ka),
                      'empty_expression':int(not expr),'reference_identity_retained':int('concept:{}'.format(int(plan.true_concept_id)) in expr),
                      'n_features_distinguishing_reference_from_root':len(expr-rootset)}
                expression_rows.append(dict(meta,expression_features_json=json.dumps(sorted(expr))))
                all_scores={m:scores_for(counts,m) for m in METHODS}
                assert competition_ranks(all_scores['jaccard']).tolist()==competition_ranks(all_scores['dice']).tolist()
                for method,scores in all_scores.items():
                    metrics=retrieval_metrics(scores,ti)
                    ancestor_better=any(scores[j]>scores[ti] and not math.isclose(scores[j],scores[ti],rel_tol=SCORE_TIE_TOLERANCE,abs_tol=SCORE_TIE_TOLERANCE) for j,cid in enumerate(ids) if cid in ancestor_ids)
                    if legacy:
                        for metric in ('top1','top5','mrr'):metrics[metric]=metrics['optimistic_'+metric]
                    trial_rows.append(dict(meta,method=method,parent_over_child=int(ancestor_better),**metrics))
                for j,cid in enumerate(ids):
                    row={'trial_id':plan.trial_id,'scenario':scenario,'condition_id':condition,'candidate_concept_id':cid,
                         'is_reference':int(j==ti),'is_ancestor':int(cid in ancestor_ids),'k':counts[j][0],'e':counts[j][1],'c':counts[j][2]}
                    row.update({m:all_scores[m][j] for m in METHODS})
                    if writer is None: writer=csv.DictWriter(score_handle,fieldnames=list(row)); writer.writeheader()
                    writer.writerow(row)
    if (ix+1)%100==0:print('Evaluated {}/{} paired base trials'.format(ix+1,len(trial_plan_df)),flush=True)
score_handle.close()
tr=pd.DataFrame(trial_rows); ex=pd.DataFrame(expression_rows)
tr.to_csv(os.path.join(OUT_DIR,'trial_results.csv.gz'),index=False,compression='gzip')
ex.to_csv(os.path.join(OUT_DIR,'simulated_expressions.csv.gz'),index=False,compression='gzip')
keys=['scenario','hierarchy_retention','attribute_retention','condition_id']
metrics=['parent_over_child','top1','top5','mrr']
summary=[]; intervals=[]
for group_key,group in tr.groupby(keys,sort=True):
    scenario,hr,ar,condition=group_key
    seed_ids=sorted(group.root_concept_id.unique()); n=len(seed_ids)
    rng=np.random.RandomState(stable_seed(RANDOM_SEED,'bootstrap',scenario,condition))
    weights=rng.multinomial(n,[1.0/n]*n,size=BOOTSTRAPS)/float(n)
    seed_values={method:group[group.method==method].groupby('root_concept_id')[metrics].mean().reindex(seed_ids).values for method in METHODS}
    for method in METHODS:
        part=group[group.method==method]; mean=part[metrics].mean(); boot=weights.dot(seed_values[method]); delta=seed_values[method]-seed_values['cosine']; bd=weights.dot(delta)
        row=dict(zip(keys,group_key));row.update(method=method,n_trials=part.trial_id.nunique(),n_seeds=n,mean_candidates=part.n_candidates.mean(),reference_tie_rate=(part.tie_size>1).mean(),empty_expression_rate=part.empty_expression.mean())
        row.update(mean.to_dict());summary.append(row)
        for j,metric in enumerate(metrics):
            lo,hi=np.percentile(boot[:,j],[2.5,97.5]); dl,dh=np.percentile(bd[:,j],[2.5,97.5])
            ci=dict(zip(keys,group_key));ci.update(method=method,metric=metric,estimate=float(mean[metric]),ci_low=float(lo),ci_high=float(hi),difference_vs_cosine=float(delta[:,j].mean()),difference_ci_low=float(dl),difference_ci_high=float(dh),bootstrap_replicates=BOOTSTRAPS,n_seeds=n)
            intervals.append(ci)
summary_df=pd.DataFrame(summary); ci_df=pd.DataFrame(intervals)
summary_df.to_csv(os.path.join(OUT_DIR,'condition_summary.csv'),index=False)
ci_df.to_csv(os.path.join(OUT_DIR,'paired_bootstrap_confidence_intervals.csv'),index=False)
for scenario in SCENARIOS:
    table=summary_df[summary_df.scenario==scenario].pivot(index=['hierarchy_retention','attribute_retention'],columns='method',values=metrics)
    table.columns=['{} | {}'.format(a,b) for a,b in table.columns]
    table.reset_index().to_csv(os.path.join(OUT_DIR,'table4_{}.csv'.format(scenario)),index=False)
# Descriptive evidence strata; does not substitute for independent clinical labels.
tr['observed_refinement_evidence']=tr.n_features_distinguishing_reference_from_root>0
strata=tr.groupby(['scenario','condition_id','method','observed_refinement_evidence'])[metrics].agg(['mean','count'])
strata.to_csv(os.path.join(OUT_DIR,'refinement_evidence_strata.csv'))
ret=ex.groupby(['scenario','hierarchy_retention','attribute_retention']).apply(lambda g:pd.Series({
 'realized_hierarchy_retention':g.hierarchy_kept.sum()/g.hierarchy_total.sum() if g.hierarchy_total.sum() else np.nan,
 'realized_attribute_retention':g.attribute_kept.sum()/g.attribute_total.sum() if g.attribute_total.sum() else np.nan,
 'empty_expression_rate':g.empty_expression.mean(),'n_expressions':len(g)}))
ret.to_csv(os.path.join(OUT_DIR,'realized_retention.csv'))
# Integrity checks: paired methods and nine conditions per scenario/base trial.
assert tr.groupby(['scenario','trial_id','condition_id']).method.nunique().eq(len(METHODS)).all()
assert ex.groupby(['scenario','trial_id']).condition_id.nunique().eq(9).all()
assert len(tr)==len(trial_plan_df)*9*len(SCENARIOS)*len(METHODS)
assert ex[ex.scenario=='primary_no_identity'].reference_identity_retained.eq(0).all()
assert np.all((tr[metrics].values>=0)&(tr[metrics].values<=1))
# Figure for primary analysis; the connected x-axis enumerates the experimental grid.
fig,axes=plt.subplots(2,2,figsize=(12,8))
p=summary_df[summary_df.scenario=='primary_no_identity']
for ax,metric,title in zip(axes.flat,metrics,['Ancestor ranked above reference','Expected Top-1','Expected Top-5','Expected reciprocal rank']):
    for method in METHODS[:4]:
        q=p[p.method==method].sort_values(['hierarchy_retention','attribute_retention'])
        ax.plot(range(9),q[metric],marker='o',label=method)
    ax.set_xticks(range(9));ax.set_xticklabels(['{:.1f}/{:.1f}'.format(h,a) for h in HIERARCHY_RETENTION_VALUES for a in ATTRIBUTE_RETENTION_VALUES],rotation=45)
    ax.set_title(title);ax.set_xlabel('Hierarchy / attribute retention');ax.set_ylabel('Proportion');ax.grid(alpha=.2)
axes[0,0].legend(fontsize=8);fig.suptitle('Corrected ontology-feature recovery without reference identity');fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR,'figure1_primary.png'),dpi=180);plt.close(fig)
metadata={'rf2_release':'US1000124_20260301','inputs':input_signatures,'n_source_rows':len(source_df),'n_unique_seeds':len(terms_df),
 'n_eligible_seeds':int(trial_plan_df.root_concept_id.nunique()),'trials_per_condition':len(trial_plan_df),
 'mean_candidates':float(trial_plan_df.n_candidates.mean()),'max_candidates':int(trial_plan_df.n_candidates.max()),
 'n_unique_evaluated_candidates':len(used_ids),'characteristic_type':CHARACTERISTIC_TYPEID,
 'scenarios':list(SCENARIOS),'methods':list(METHODS),'bootstrap_replicates':BOOTSTRAPS,'random_seed':RANDOM_SEED,
 'hierarchy_containment_fraction':float(audit_df.hierarchy_contained.mean()),'full_containment_fraction':float(audit_df.full_features_contained.mean()),
 'software':{'python':sys.version,'numpy':np.__version__,'pandas':pd.__version__},'completed_utc':datetime.utcnow().isoformat()+'Z'}
json.dump(metadata,open(os.path.join(OUT_DIR,'run_metadata.json'),'w'),indent=2)
print(json.dumps(metadata,indent=2))
print(summary_df[(summary_df.scenario=='primary_no_identity') & summary_df.method.isin(['cosine','tversky_0.1'])].to_string(index=False))


## Interpretation

Use primary_no_identity for the main results. Report identity_retained and legacy_same_release as sensitivity analyses. Confidence intervals resample the 71-or-other contributing seed concepts, retaining all repeated trials and paired methods; overlapping ontology families may still induce residual dependence. Parameter values are prespecified comparisons, not independently validated tuning. Jaccard and Dice induce identical rankings and therefore are included as a useful implementation cross-check, not independent evidence. The source release differs from the previously attached results; changes cannot all be attributed to code corrections.
